<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/4.2%20Building%20RAGs%20and%20Multi-Step%20Chains/4%20Tutorial%20-%20Multi%E2%80%91Step%20Chain%20(Summary%20%2B%20Q%26A)%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### OpenRouter API Key

This notebook uses OpenRouter through LangChain's OpenAI-compatible interface. Enter your OpenRouter API key when prompted. You do not need a separate OpenAI API key.


In [1]:
# Install pinned dependencies (Colab-ready; safe to re-run).
# Based on latest compatible versions
!pip install -q langchain-core==1.6.3 langchain-openai==1.6.2 langchain-pinecone==0.2.13 pinecone==7.3.0 python-dotenv==1.2.3 tiktoken==0.14.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.2 MB/s eta 0:00:00


## Tutorial: Multi‑Step Chain with Retrieval (Summary + Q&A)
We’ll optionally retrieve context from Pinecone, summarize it, then answer the question from the summary.


In [2]:
import os, time
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter OPENROUTER_API_KEY (hidden): ")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or getpass("Enter PINECONE_API_KEY (hidden): ")

# PineconeVectorStore reads the key from the environment rather than a variable.
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
MODEL = "openai/gpt-4.1-mini"

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=MODEL, api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1", temperature=0, seed=42)
embeddings = OpenAIEmbeddings(model="openai/text-embedding-3-small", api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1", dimensions=1536)
INDEX_NAME = os.getenv("PINECONE_INDEX", "lc-demo-index")

pc = Pinecone(api_key=PINECONE_API_KEY)
# Create the index when running this notebook on its own, not after tutorial 2.
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=os.getenv("PINECONE_REGION", "us-east-1"))
    )

# A newly created index is not queryable until it reports ready.
while not pc.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)

vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


Enter OPENROUTER_API_KEY (hidden): ··········
Enter PINECONE_API_KEY (hidden): ··········


### Step 1: Define subchain prompts
- `summary_chain`: compresses context into 3 bullets
- `qa_chain`: answers from the summary only
- (Optional) `retriever`: pulls top‑k docs to form the context if none is provided


In [4]:
summary_prompt = PromptTemplate.from_template(
    "Summarize the following text into 3 bullet points:\n{context}\n\nBullets:"
)
qa_prompt = PromptTemplate.from_template(
    "Using ONLY the SUMMARY below, answer the QUESTION.\n"
    "If not answerable, say: I don't know.\n\n"
    "SUMMARY:\n{summary}\n\n"
    "QUESTION: {question}\n"
    "ANSWER:"
)

summary_chain = summary_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm | StrOutputParser()


### Step 2: Compose multi-step pipeline
No agents: deterministic steps for speed and testability.


In [6]:
def summarize_then_answer(question: str, context: str = "") -> str:
    if not context:
        # Optional: pull from Pinecone when context not provided
        docs = retriever.invoke(question)
        context = "\n\n".join(d.page_content for d in docs)
    summary = summary_chain.invoke({"context": context})
    answer = qa_chain.invoke({"summary": summary, "question": question})
    return answer

kb = (
    "LangChain is a framework for building with LLMs. It offers prompts, chains, tools, "
    "and agents to compose complex workflows and integrate with data sources."
)
print(summarize_then_answer("Name two abstractions LangChain provides.", context=kb))
print(summarize_then_answer("What enables retrieval in this setup?"))


Prompts and chains.
Prompt templates and retrievers enable retrieval in this setup.
